In [1]:
!pip install  -U langchain

In [3]:
!pip install langchain_groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [langchain_groq]


In [2]:
from google.colab import userdata
GROQ_API_KEY=userdata.get('GROQ_API_KEY')

In [7]:

import os
import asyncio
from typing import Optional

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import (
    Runnable,
    RunnableParallel,
    RunnablePassthrough
)

# --- Configuration ---

try:
    llm: Optional[ChatGroq] = ChatGroq(
        model="openai/gpt-oss-120b",
        temperature=0.7,
        api_key=GROQ_API_KEY
    )

except Exception as e:
    print(f"Error initializing language model: {e}")
    llm = None


# --- Define Independent Chains ---

# 1. Summary Chain
summarize_chain: Runnable = (
    ChatPromptTemplate.from_messages([
        ("system", "Summarize the following topic concisely:"),
        ("user", "{topic}")
    ])
    | llm
    | StrOutputParser()
)


# 2. Questions Chain
questions_chain: Runnable = (
    ChatPromptTemplate.from_messages([
        ("system", "Generate three interesting questions about the following topic:"),
        ("user", "{topic}")
    ])
    | llm
    | StrOutputParser()
)


# 3. Key Terms Chain
terms_chain: Runnable = (
    ChatPromptTemplate.from_messages([
        ("system", "Identify 5-10 key terms from the following topic, separated by commas:"),
        ("user", "{topic}")
    ])
    | llm
    | StrOutputParser()
)


# --- Build Parallel Chain ---

map_chain = RunnableParallel(
    {
        "summary": summarize_chain,
        "questions": questions_chain,
        "key_terms": terms_chain,
        "topic": RunnablePassthrough()
    }
)


# --- Synthesis Prompt ---

synthesis_prompt = ChatPromptTemplate.from_messages([
    ("system", """
    Based on the following information:

    Summary: {summary}

    Related Questions: {questions}

    Key Terms: {key_terms}

    Synthesize a comprehensive answer.
    """),
    ("user", "Original topic: {topic}")
])


# --- Full Parallel Chain ---

full_parallel_chain = (
    map_chain
    | synthesis_prompt
    | llm
    | StrOutputParser()
)


# --- Run the Chain ---

async def run_parallel_example(topic: str) -> None:

    if not llm:
        print("LLM not initialized. Cannot run example.")
        return

    print(f"\n--- Running Parallel LangChain Example ---")
    print(f"Topic: {topic}")

    try:
        response = await full_parallel_chain.ainvoke(
            {"topic": topic}
        )

        print("\n--- Final Response ---")
        print(response)

    except Exception as e:
        print(f"\nAn error occurred: {e}")


if __name__ == "__main__":

  test_topic = "The history of space exploration"
  # Use await directly instead of asyncio.run because notebooks already run an event loop
  await run_parallel_example(test_topic)



--- Running Parallel LangChain Example ---
Topic: The history of space exploration

--- Final Response ---
## The History of Space Exploration – A Concise Synthesis  

Below is a **structured, narrative answer** that weaves together the timeline, the geopolitical forces, the unsung contributors, and the cultural‑media feedback loop that have shaped humanity’s journey from the first theoretical rockets to today’s multi‑national, commercial‑driven push toward the Moon, Mars and beyond.

---

### 1.  Cold‑War Rivalry → the Foundations of Modern Space Collaboration  

| Phase | Cold‑War Driver | Signature Milestones | Lasting Institutional Legacy |
|-------|----------------|----------------------|------------------------------|
| **Early Competition (1950‑1962)** | *Ideological showcase*: the United States and the USSR wanted to prove the superiority of capitalism vs. communism through “technological might.” | • **Sputnik 1** (1957) – first artificial satellite, shocked the world and spur